# Export LidarScout Model to ONNX
This notebook captures the `IpesCnn` model and exports it to an `.onnx` file for C++ inference using ONNX Runtime.

In [9]:
import os
import torch
import pytorch_lightning as pl

# Import your specific model class
from source.modules.ipes_cnn import IpesCnn

In [10]:
CHECKPOINT_PATH = r"C:\repos\lidarscout_training\models\ipes_cnn_v2_extra_data_voidloss_arch2\alpha\checkpoints\last.ckpt"
OUTPUT_ONNX_PATH = r"exported_models\ipes_cnn_v2_extra_data_voidloss_arch2\ipes_cnn_rgb.onnx"
YAML_CONFIG_PATH = r"C:\repos\lidarscout_training\models\ipes_cnn_v2_extra_data_voidloss_arch2\alpha\config.yaml"

In [11]:
print(f"Loading config from {YAML_CONFIG_PATH}...")

with open(YAML_CONFIG_PATH, 'r') as file:
    import yaml
    config = yaml.safe_load(file)
    
model_kwargs = config['model']['init_args']
data_kwargs = config['data']['init_args']

model_kwargs['pts_to_img_methods'] = data_kwargs['pts_to_img_methods']
model_kwargs['hm_size'] = data_kwargs['hm_size']
model_kwargs['in_file'] = data_kwargs['in_file']

if model_kwargs.get('workers') is None:
    model_kwargs['workers'] = data_kwargs.get('workers', 0)

Loading config from C:\repos\lidarscout_training\models\ipes_cnn_v2_extra_data_voidloss_arch2\alpha\config.yaml...


In [ ]:
import inspect


print(f"Loading model from {CHECKPOINT_PATH}...")

network = IpesCnn.load_from_checkpoint(
    checkpoint_path=CHECKPOINT_PATH,
    **model_kwargs
)

network.eval()
print("Model loaded and set to eval mode successfully!")
print(inspect.signature(network.forward))

Loading model from C:\repos\lidarscout_training\models\ipes_cnn_v2_extra_data_voidloss_arch2\alpha\checkpoints\last.ckpt...
Model loaded and set to eval mode successfully!
(batch)


In [13]:
print("Applying cuDNN deterministic settings...")
torch.backends.cudnn.enabled = True
torch.backends.cudnn.allow_tf32 = False
torch.backends.cudnn.benchmark = True
torch.backends.cudnn.deterministic = False

Applying cuDNN deterministic settings...


In [14]:
def make_example_data(net, batch_size=1):
    res = net.hm_interp_size
    example_inputs = dict()

    for m in net.input_methods:
        example_inputs[f'patch_hm_{m}'] = torch.rand(batch_size, 1, res, res)
        example_inputs[f'patch_rgb_{m}'] = torch.rand(batch_size, 3, res, res)
        
    example_inputs['patch_hm_mask'] = torch.rand(batch_size, 1, res, res)
    
    return example_inputs

dummy_inputs = make_example_data(network, batch_size=1)

# ONNX export uses the exact same argument structure as FX Tracing
export_args = (dummy_inputs,)

print("Dummy inputs generated and formatted for export.")

Dummy inputs generated and formatted for export.


In [15]:
print(f"\nExporting model to {OUTPUT_ONNX_PATH}")
os.makedirs(os.path.dirname(OUTPUT_ONNX_PATH), exist_ok=True)

try:
    # PyTorch to ONNX export
    torch.onnx.export(
        network,
        args=(),              # empty positional args
        kwargs={'batch': dummy_inputs},   # dict delivered as the 'batch' kwarg
        f=OUTPUT_ONNX_PATH,
        export_params=True,
        opset_version=18,     # use 18; your torch warns it's upgrading 17→18 anyway
        do_constant_folding=True,
        input_names=list(dummy_inputs.keys()),
        output_names=['output'],
    )
    print("1. Exported to ONNX format successfully.")
    
except Exception as e:
    print("Export failed!")
    raise e

print(f"\nExport complete! Model saved to {OUTPUT_ONNX_PATH}")


Exporting model to exported_models\ipes_cnn_v2_extra_data_voidloss_arch2\ipes_cnn_rgb.onnx
[torch.onnx] Obtain model graph for `IpesCnn([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `IpesCnn([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.12_3.12.2800.0_x64__qbz5n2kfra8p0\Lib\copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
1. Exported to ONNX format successfully.

Export complete! Model saved to exported_models\ipes_cnn_v2_extra_data_voidloss_arch2\ipes_cnn_rgb.onnx


In [16]:
try:
    import onnx
    print("Validating ONNX graph...")
    onnx_model = onnx.load(OUTPUT_ONNX_PATH)
    onnx.checker.check_model(onnx_model)
    print("Validation successful! The model is ready for C++.")
except ImportError:
    print("\nNote: 'onnx' Python package is not installed. Skipping graph validation.")
    print("You can install it via 'pip install onnx' if you want to verify the output file.")


Validating ONNX graph...
Validation successful! The model is ready for C++.
